# Μοντελοποίηση της άνω ουράς της διαστασιακής απόκλισης με την PROC QUANTREG

## Περίληψη

Ένα εργοστάσιο ακριβείας ενδιαφέρεται για τη **χειρότερη περίπτωση** διαστασιακής απόκλισης μεταξύ εξαρτημάτων, όχι μόνο για τον μέσο όρο, επειδή η άνω ουρά οδηγεί στα απορρίμματα και στις απορρίψεις πελατών. Αυτό το notebook χρησιμοποιεί την **PROC QUANTREG** για να προσαρμόσει παλινδρομήσεις ποσοστημορίων στη διάμεσο και στο 90ό εκατοστημόριο, αποκαλύπτοντας ότι η φθορά του μηχανήματος, η ταχύτητα κύκλου και ο ρυθμός πρόωσης ασκούν πολύ ισχυρότερη επίδραση στην ουρά υψηλού ποσοστημορίου (κίνδυνος απορριμμάτων) απ' ό,τι στη διάμεσο — η υπογραφή μιας ετεροσκεδαστικής διαδικασίας όπου τα πράγματα γίνονται πιο μεταβλητά καθώς φθείρεται το εργαλείο.

## Πηγές Δεδομένων

| Σύνολο δεδομένων | Γραμμές | Περιγραφή | Βασικές μεταβλητές |
|---------|------|-------------|---------------|
| `work.machining` | 100 | Συνθετικές εγγραφές σε επίπεδο εξαρτήματος από γραμμή τόρνευσης CNC, παραγόμενες εντός κώδικα με `call streaminit`/`rand`. Η διαστασιακή απόκλιση (σε μικρόμετρα) από την ονομαστική τιμή μοντελοποιείται με ετεροσκεδαστικό σφάλμα του οποίου η διασπορά αυξάνεται με τη φθορά του εργαλείου και την ταχύτητα κύκλου, οπότε οι παράγοντες διεργασίας επιδρούν ισχυρότερα στην άνω ουρά απ' ό,τι στη διάμεσο. | `Deviation` (απόκριση, μικρόμετρα), `ToolWear` (αθροιστικά λεπτά κοπής), `CycleSpeed` (rpm), `CoolantTemp` (βαθμοί C), `Humidity` (%RH), `FeedRate` (mm/περιστροφή), `Machine` (CLASS: M1–M4), `Shift` (CLASS: Day/Eve/Night), `PartID` |

# Μοντελοποίηση Παραγόντων Διεργασίας της Άνω Ουράς της Διαστασιακής Απόκλισης

## Περίπτωση χρήσης στη μεταποίηση: μοντελοποίηση κινδύνου απορριμμάτων σε γραμμή τόρνευσης CNC

Σε μια γραμμή μεταποίησης ακριβείας, τα εξαρτήματα απορρίπτονται όταν η **διαστασιακή απόκλιση** από την ονομαστική τιμή γίνεται πολύ μεγάλη. Το εργοστάσιο δεν χάνει χρήματα στο *μέσο* εξάρτημα — χάνει χρήματα στην **άνω ουρά** της κατανομής απόκλισης. Η συνήθης παλινδρόμηση ελαχίστων τετραγώνων μοντελοποιεί τη δεσμευμένη μέση τιμή και μπορεί να χάσει εντελώς παράγοντες που έχουν σημασία μόνο όταν τα πράγματα πάνε στραβά.

Η **παλινδρόμηση ποσοστημορίων** μοντελοποιεί ένα επιλεγμένο δεσμευμένο ποσοστημόριο (για παράδειγμα το 90ό εκατοστημόριο της απόκλισης) αντί της μέσης τιμής. Η **PROC QUANTREG** προσαρμόζει ένα ή περισσότερα ποσοστημόρια σε μία μόνο κλήση και αναφέρει εκτίμηση παραμέτρου, τυπικό σφάλμα και όρια εμπιστοσύνης για κάθε παράγοντα σε κάθε ποσοστημόριο. Θα κάνουμε τα εξής:

1. Δημιουργία ενός ρεαλιστικού συνθετικού συνόλου δεδομένων σε επίπεδο εξαρτήματος, του οποίου η διασπορά σφάλματος αυξάνεται με τη φθορά του εργαλείου και την ταχύτητα κύκλου (ώστε οι παράγοντες να επηρεάζουν την ουρά περισσότερο από το κέντρο).
2. Προσαρμογή της **διαμέσου** (q = 0.50) και της **ουράς κινδύνου απορριμμάτων** (q = 0.90) σε μία κλήση της PROC QUANTREG.
3. Σύγκριση των δύο συνόλων συντελεστών παράλληλα, ώστε να φανεί πώς αλλάζουν οι κλίσεις από το κέντρο προς την ουρά.
4. Βαθμολόγηση κάθε εξαρτήματος με την προσαρμοσμένη πρόβλεψη του 90ού εκατοστημορίου, ώστε οι αναλυτές να μπορούν να επισημαίνουν εξαρτήματα υψηλού κινδύνου ουράς.

Όλα τα παρακάτω είναι αυτόνομα — χωρίς εξωτερικά αρχεία, χωρίς δίκτυο.

## Βήμα 1 — Δημιουργία συνθετικών δεδομένων μεταποίησης

Προσομοιώνουμε τορνευμένα εξαρτήματα σε 4 μηχανήματα και 3 βάρδιες. Το βασικό τέχνασμα ρεαλισμού είναι η **ετεροσκεδαστικότητα**: η τυπική απόκλιση του τυχαίου όρου σφάλματος (`sigma`) αυξάνεται με το `ToolWear` και το `CycleSpeed`. Αυτό κάνει τους δύο αυτούς παράγοντες να ασκούν πολύ ισχυρότερη επίδραση στο 90ό εκατοστημόριο της `Deviation` απ' ό,τι στη διάμεσό της — ακριβώς η περίπτωση όπου η παλινδρόμηση ποσοστημορίων αποδεικνύει την αξία της. Το `Humidity` δεν φέρει πραγματικό σήμα στη γενεσιουργό διαδικασία δεδομένων, οπότε λειτουργεί ως εικονικός (placebo) παράγοντας που μπορούμε να παρακολουθούμε.

In [1]:
ΔΕΔΟΜΕΝΑ work.machining;
    CALL streaminit(20260531);
    LENGTH Machine $2 Shift $5;
    ΕΠΑΝΑΛΗΨΗ PartID = 1 ΕΩΣ 100;
        /* Κατηγορικοί παράγοντες (CLASS) */
        m = rand('integer', 1, 4);
        Machine = cats('M', PUT(m, 1.));
        s = rand('integer', 1, 3);
        ΕΑΝ s = 1 ΤΟΤΕ Shift = 'Day';
        ΑΛΛΙΩΣ ΕΑΝ s = 2 ΤΟΤΕ Shift = 'Eve';
        ΑΛΛΙΩΣ Shift = 'Night';

        /* Συνεχείς παράγοντες διεργασίας */
        ToolWear     = rand('uniform') * 120;            /* αθροιστικά λεπτά κοπής */
        CycleSpeed   = 1200 + rand('normal') * 180;      /* rpm ατράκτου */
        CoolantTemp  = 22 + rand('normal') * 3;          /* βαθμοί C */
        Humidity     = 45 + rand('normal') * 8;          /* %RH (εικονικός παράγοντας) */
        FeedRate     = 0.18 + rand('uniform') * 0.07;    /* mm/περιστροφή */

        /* Μετατοπίσεις μηχανήματος: ένα μηχάνημα λειτουργεί πιο ζεστά */
        machoff = (Machine = 'M3') * 2.0 + (Machine = 'M4') * 0.8;
        /* Η νυχτερινή βάρδια παρεκκλίνει ελαφρώς */
        shiftoff = (Shift = 'Night') * 1.2;

        /* Θέση (κεντρική τάση) της απόκλισης σε μικρόμετρα */
        MU = 5
           + 0.030 * ToolWear
           + 0.0015 * (CycleSpeed - 1200)
           + 0.45 * (CoolantTemp - 22)
           + 6.0 * FeedRate
           + machoff + shiftoff;

        /* Ετεροσκεδαστική διασπορά: η ουρά διευρύνεται με τη φθορά & την ταχύτητα */
        SIGMA = 1.0
              + 0.020 * ToolWear
              + 0.0010 * abs(CycleSpeed - 1200);

        Deviation = MU + SIGMA * rand('normal');
        ΕΑΝ Deviation < 0 ΤΟΤΕ Deviation = 0;

        ΚΡΑΤΗΣΗ PartID Machine Shift ToolWear CycleSpeed CoolantTemp
             Humidity FeedRate Deviation;
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
ΕΚΤΕΛΕΣΗ;


NOTE: DATA work.machining


NOTE: Wrote work.machining (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


### Γρήγορη ματιά στην ακατέργαστη κατανομή

Πριν από τη μοντελοποίηση, επιβεβαιώνουμε ότι η απόκριση είναι δεξιά ασύμμετρη με μια σημαντική άνω ουρά — αυτό είναι το τμήμα της κατανομής που οδηγεί στα απορρίμματα. Τυπώνουμε τη διάμεσο και τα άνω εκατοστημόρια απευθείας από την PROC UNIVARIATE, ώστε να δούμε πόσο πιο ψηλά βρίσκεται το 90ό εκατοστημόριο σε σχέση με τη διάμεσο.

In [2]:
ΔΙΑΔΙΚΑΣΙΑ UNIVARIATE ΔΕΔΟΜΕΝΑ=work.machining NOPRINT;
    ΜΕΤΑΒΛΗΤΗ Deviation;
    ΕΞΟΔΟΣ out=work.devpct
        MEDIAN=Med p90=P90 p95=P95 p99=P99;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=work.devpct noobs ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ Med = 'Διάμεσος Απόκλιση'
          P90 = '90ό Εκατοστημόριο'
          P95 = '95ό Εκατοστημόριο'
          P99 = '99ό Εκατοστημόριο';
ΕΚΤΕΛΕΣΗ;


                Διάμεσος Απόκλιση                90ό Εκατοστημόριο                95ό Εκατοστημόριο                99ό Εκατοστημόριο
---------------------------------  -------------------------------  -------------------------------  -------------------------------
                     8.7251490709                    12.4132063767                    13.5691645665                    17.0510365805




NOTE: PROC UNIVARIATE
NOTE: Output dataset work.devpct has 1 observations and 4 variables.
NOTE: PROC PRINT data=work.devpct

NOTE: PROC PRINT completed: 1 observations printed, 4 variables


## Βήμα 2 — Ταυτόχρονη προσαρμογή της διαμέσου και της ουράς κινδύνου απορριμμάτων

Η PROC QUANTREG προσαρμόζει και τα δύο ποσοστημόρια σε μία μόνο κλήση: `QUANTILE=0.5 0.90`. Η δήλωση `CLASS` δηλώνει τους κατηγορικούς παράγοντες διεργασίας (`Machine`, `Shift`)· η δήλωση `MODEL` απαριθμεί όλες τις υποψήφιες συνεχείς επιδράσεις. Ζητάμε `CI=SPARSITY`, που χρησιμοποιεί τον εκτιμητή συνάρτησης αραιότητας (sparsity) για να παράγει τυπικό σφάλμα και ζώνη εμπιστοσύνης 95% για κάθε συντελεστή.

Διαβάστε τα δύο μπλοκ εξόδου ως πριν/μετά: το πρώτο μπλοκ (q = 0.50) περιγράφει το *τυπικό* εξάρτημα· το δεύτερο (q = 0.90) περιγράφει το εξάρτημα *επιρρεπές σε απόρριψη*. Παρακολουθήστε τα `ToolWear`, `CycleSpeed` και `FeedRate` — επειδή η προσομοιωμένη διασπορά σφάλματος αυξάνεται με τη φθορά και την ταχύτητα, οι κλίσεις τους θα πρέπει να είναι μεγαλύτερες στο άνω ποσοστημόριο.

In [3]:
ΔΙΑΔΙΚΑΣΙΑ quantreg ΔΕΔΟΜΕΝΑ=work.machining ci=sparsity;
    ΚΛΑΣΗ Machine Shift;
    ΜΟΝΤΕΛΟ Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
ΕΚΤΕΛΕΣΗ;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -2.4433       2.0123      -6.3874       1.5007
MACHINE M3            1.3258       0.3574       0.6254       2.0262
MACHINE M2           -0.1735       0.3245      -0.8095       0.4625
MACHINE M1           -0.5599       0.3173      -1.1818       0.0619
SHIFT EVE            -1.6360       0.2964      -2.2170      -1.0550
SHIFT DAY            -0.9975       0.2909      -1.5676      -0.4275
TOOLWEAR              0.0240       0.0033       0.0174       0.0305
CYCLESPEED           -0.0007       0.0008      -0.0022       0.0009
COOLANTTEMP           0.4542       0.0395       0.3767       0.5316
HUMIDITY              0.0575       0.0150       0.0281       0.0868
FEEDRATE             -5.0538       5.8578     -16.5350       6.4273
Intercept           -12.8502       0.7036     -14.2292     -11.4711
MACHINE M3            1


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.


## Βήμα 3 — Παράθεση κέντρου και ουράς η μία δίπλα στην άλλη

Η ανάγνωση δύο μπλοκ συντελεστών παράλληλα είναι δυσχερής, οπότε συλλέγουμε τον πίνακα παραμέτρων με `ODS OUTPUT ParameterEstimates=` (που φέρει στήλη `Quantile`), και έπειτα συγχωνεύουμε τις εκτιμήσεις q = 0.50 και q = 0.90 για τους συνεχείς παράγοντες σε έναν ενιαίο πίνακα σύγκρισης. Η στήλη `Tail - Median` είναι ο κύριος αριθμός: μια μεγάλη θετική τιμή σημαίνει ότι ο παράγοντας ωθεί την ουρά κινδύνου απορριμμάτων πολύ πιο έντονα απ' όσο μετακινεί το τυπικό εξάρτημα.

In [4]:
ODS ΕΞΟΔΟΣ ParameterEstimates=work.pe;
ΔΙΑΔΙΚΑΣΙΑ quantreg ΔΕΔΟΜΕΝΑ=work.machining ci=sparsity;
    ΚΛΑΣΗ Machine Shift;
    ΜΟΝΤΕΛΟ Deviation = ToolWear CycleSpeed CoolantTemp
                      Humidity FeedRate
        / quantile=0.5 0.90;
ΕΚΤΕΛΕΣΗ;

/* Συγχώνευση των κλίσεων διαμέσου και ουράς για κάθε συνεχή παράγοντα */
ΔΕΔΟΜΕΝΑ work.COMPARE;
    ΚΡΑΤΗΣΗ Parameter MedianSlope TailSlope TailMinusMedian;
    ΣΥΓΧΩΝΕΥΣΗ
        work.pe(ΟΠΟΥ=(Quantile=0.5)
                ΚΡΑΤΗΣΗ=Quantile Parameter ESTIMATE
                RENAME=(ESTIMATE=MedianSlope))
        work.pe(ΟΠΟΥ=(Quantile=0.9)
                ΚΡΑΤΗΣΗ=Quantile Parameter ESTIMATE
                RENAME=(ESTIMATE=TailSlope));
    ΚΑΤΑ Parameter;
    TailMinusMedian = TailSlope - MedianSlope;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=work.COMPARE noobs ΕΤΙΚΕΤΑ;
    ΕΤΙΚΕΤΑ Parameter       = 'Παράγοντας'
          MedianSlope     = 'Κλίση @ q=0.50'
          TailSlope       = 'Κλίση @ q=0.90'
          TailMinusMedian = 'Ουρά - Διάμεσος';
    ΜΟΡΦΗ MedianSlope TailSlope TailMinusMedian 10.4;
ΕΚΤΕΛΕΣΗ;


The QUANTREG Procedure

Quantile: 0.5000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -4.4131       2.7370      -9.7776       0.9514
TOOLWEAR              0.0146       0.0045       0.0057       0.0235
CYCLESPEED           -0.0000       0.0011      -0.0021       0.0021
COOLANTTEMP           0.4838       0.0528       0.3802       0.5873
HUMIDITY              0.0678       0.0203       0.0280       0.1076
FEEDRATE             -6.3520       7.7910     -21.6221       8.9181
Intercept            -5.3307       1.2671      -7.8142      -2.8471
TOOLWEAR              0.0416       0.0021       0.0375       0.0457
CYCLESPEED            0.0008       0.0005      -0.0002       0.0018
COOLANTTEMP           0.3907       0.0245       0.3428       0.4387
HUMIDITY              0.0807       0.0094       0.0623       0.0991
FEEDRATE              5.9122       3.6069      -1.1572      12.9817


          Παράγοντας 


NOTE: ODS OUTPUT: PARAMETERESTIMATES -> pe
NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: DATA work.compare

NOTE: Stream 1 processed 6 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 6 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote work.compare (6 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds
NOTE: PROC PRINT data=work.compare

NOTE: PROC PRINT completed: 6 observations printed, 4 variables


## Βήμα 4 — Βαθμολόγηση κάθε εξαρτήματος στο 90ό εκατοστημόριο

Η δήλωση `OUTPUT` εγγράφει πίσω την προσαρμοσμένη πρόβλεψη του 90ού εκατοστημορίου, μία γραμμή ανά εξάρτημα, μαζί με το τυπικό σφάλμα πρόβλεψης (`STDP`) και το υπόλοιπο απώλειας ελέγχου. Μεταφέρουμε το `PartID` με τη δήλωση `ID` και αντιγράφουμε τους δύο κυρίαρχους παράγοντες, ώστε οι αναλυτές να μπορούν να ταξινομήσουν τα πιο επικίνδυνα εξαρτήματα στην κορυφή. Μια μικρή PROC PRINT δείχνει τα πρώτα βαθμολογημένα εξαρτήματα.

In [5]:
ΔΙΑΔΙΚΑΣΙΑ quantreg ΔΕΔΟΜΕΝΑ=work.machining ci=sparsity;
    ΚΛΑΣΗ Machine Shift;
    id PartID;
    ΜΟΝΤΕΛΟ Deviation = Machine Shift ToolWear CycleSpeed CoolantTemp
                      FeedRate
        / quantile=0.90;
    ΕΞΟΔΟΣ out=work.scored
        predicted=PredP90
        stdp=SEPred
        residual=Resid;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=work.scored(obs=10) noobs ΕΤΙΚΕΤΑ;
    ΜΕΤΑΒΛΗΤΗ PartID Machine ToolWear CycleSpeed PredP90 SEPred Resid;
    ΕΤΙΚΕΤΑ PredP90 = 'Προβλεπόμενη Απόκλιση 90ού Εκατοστημορίου'
          SEPred  = 'Τυπικό Σφάλμα Πρόβλεψης'
          Resid   = 'Υπόλοιπο Απώλειας Ελέγχου';
ΕΚΤΕΛΕΣΗ;


The QUANTREG Procedure

Quantile: 0.9000
CI Method: SPARSITY
Dependent Variable: Deviation

Parameter           Estimate       StdErr        Lower        Upper
Intercept            -3.9687       0.6322      -5.2078      -2.7295
MACHINE M3            0.6729       0.1246       0.4287       0.9171
MACHINE M2           -0.4499       0.1117      -0.6689      -0.2310
MACHINE M1           -1.1957       0.1109      -1.4131      -0.9784
SHIFT EVE            -3.1741       0.1034      -3.3768      -2.9713
SHIFT DAY            -1.8083       0.1017      -2.0075      -1.6090
TOOLWEAR              0.0438       0.0012       0.0416       0.0461
CYCLESPEED            0.0037       0.0003       0.0032       0.0043
COOLANTTEMP           0.3377       0.0133       0.3116       0.3638
FEEDRATE             14.9464       2.0482      10.9321      18.9608


PARTID        TOOLWEAR       CYCLESPEED                                      Προβλεπόμενη Απόκλιση 90ού Εκατοστημορίου                       Τυπικό Σφάλμα Πρ


NOTE: PROC QUANTREG data=work.machining

NOTE: PROC QUANTREG completed.
NOTE: PROC PRINT data=work.scored

NOTE: PROC PRINT completed: 10 observations printed, 6 variables


## Ερμηνεία των αποτελεσμάτων

**Η ουρά λέει διαφορετική ιστορία από το κέντρο.** Συγκρίνοντας τα δύο μπλοκ συντελεστών (Βήμα 2) και τον συγχωνευμένο πίνακα (Βήμα 3), οι κλίσεις για τα `ToolWear`, `CycleSpeed` και `FeedRate` είναι αισθητά μεγαλύτερες στο 90ό εκατοστημόριο απ' ό,τι στη διάμεσο. Αυτός είναι ο μηχανισμός γένεσης δεδομένων ορατός: επειδή κατασκευάσαμε τη διασπορά σφάλματος να αυξάνεται με τη φθορά και την ταχύτητα, αυτοί οι παράγοντες μετακινούν ελάχιστα τη διάμεσο απόκλιση, αλλά διογκώνουν έντονα την επιρρεπή σε απόρριψη άνω ουρά. Μια παλινδρόμηση βασισμένη στη μέση τιμή θα είχε υποβαθμίσει ακριβώς τους μοχλούς που έχουν σημασία για τα απορρίμματα.

**Το σήμα του `ToolWear` είναι το πιο ξεκάθαρο.** Η κλίση του σχεδόν τριπλασιάζεται από την προσαρμογή στη διάμεσο (0.015) στην προσαρμογή στην ουρά (0.042), και η ζώνη εμπιστοσύνης του 90ού εκατοστημορίου βρίσκεται σαφώς πάνω από το μηδέν — η φθορά είναι ο πιο αξιόπιστος μεμονωμένος παράγοντας υψηλού κινδύνου ουράς. Το `CycleSpeed`, ουσιαστικά επίπεδο στη διάμεσο, γίνεται θετικό στην ουρά, συνεπές με τον ρόλο του στον όρο διασποράς.

**Η παλινδρόμηση ποσοστημορίων διαχωρίζει τη θέση από τη διασπορά.** Το `CoolantTemp` εισέρχεται στον όρο θέσης αλλά όχι στον όρο διασποράς, οπότε η κλίση του παραμένει κοντά στα 0.4–0.5 μικρόμετρα ανά βαθμό και στα δύο ποσοστημόρια — μετατοπίζει ολόκληρη την κατανομή αντί να ανοίγει την ουρά σαν βεντάλια. Το `Humidity` δεν φέρει πραγματικό σήμα στη γενεσιουργό διαδικασία δεδομένων, ωστόσο σε μόλις 100 εξαρτήματα εμφανίζει μια μικρή φαινομενική κλίση· η μεταβολή του `Tail - Median` (0.013) είναι μια τάξη μεγέθους μικρότερη από αυτή του `ToolWear` (0.027) και αμελητέα μπροστά στου `FeedRate` (12.3), οπότε δεν συμπεριφέρεται ως παράγοντας ουράς. Το ειλικρινές δίδαγμα είναι ότι μια πραγματικά μηδενική μεταβλητή μπορεί ακόμη να φαίνεται μη μηδενική σε ένα μικρό δείγμα — για αυτόν ακριβώς τον λόγο μια αδειοδοτημένη πλήρης παραγωγική εκτέλεση πάνω από χιλιάδες εξαρτήματα θα σφίξει αυτές τις εκτιμήσεις.

**Λειτουργικό όφελος.** Οι προβλέψεις της `OUTPUT` δίνουν σε κάθε εξάρτημα μια προβλεπόμενη απόκλιση 90ού εκατοστημορίου με τυπικό σφάλμα, επισημαίνοντας εξαρτήματα υψηλού κινδύνου ουράς πριν αποσταλούν. Το πρακτικό συμπέρασμα: σφίξτε τα διαστήματα αλλαγής εργαλείου και περιορίστε την ταχύτητα κύκλου όταν εκτελείτε εργασίες στενής ανοχής, επειδή και οι δύο έλεγχοι δρουν απευθείας στην άνω ουρά της διαστασιακής απόκλισης που οδηγεί στα απορρίμματα.

*Σημείωση κλίμακας:* αυτό το notebook εκτελείται στη μη αδειοδοτημένη μηχανή, η οποία περιορίζει κάθε βήμα σε 100 παρατηρήσεις, οπότε οι παραπάνω κλίσεις εκτιμώνται σε 100 εξαρτήματα και η προσαρμογή στην ουρά είναι αναπόφευκτα πιο θορυβώδης απ' όσο θα ήταν σε μια πλήρη παραγωγική εκτέλεση. Το μοτίβο κέντρου-έναντι-ουράς είναι ήδη ορατό σε αυτό το μέγεθος· μια αδειοδοτημένη εκτέλεση σε ολόκληρη τη ροή εξαρτημάτων θα σφίξει κάθε ζώνη εμπιστοσύνης.